In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns   
import pandas as pd
from scipy import stats

df = pd.read_csv('../../../Data/processed/cleaned_customer_data.csv')

## Loyalty and Brand Relationship EDA
Goal: Evaluate how loyalty signals relate to churn using statistical testing and effect size interpretation.

### Scope
1. `Membership_Years`
2. `Payment_Method_Diversity`
3. `Wishlist_Items`

### Workflow
1. Correlation screening with churn
2. Distribution comparison by churn group
3. Mann-Whitney U test + rank-biserial effect size
4. Executive insight summary and export

In [ ]:
Loyalty_features = [
    "Membership_Years",
    "Payment_Method_Diversity",
    "Wishlist_Items"
]

In [ ]:
corr_loyalty = df[Loyalty_features].corrwith(df["Churned"]).sort_values()
corr_loyalty.to_frame("corr_with_churn")

Wishlist_Items             -0.189569
Membership_Years           -0.000623
Payment_Method_Diversity    0.004767
dtype: float64

## Loyalty Feature Analysis

Initial evidence suggests loyalty features have weaker predictive power than engagement and purchase behavior signals.
- `Wishlist_Items` often shows a mild negative association with churn.
- `Membership_Years` and `Payment_Method_Diversity` are usually weaker standalone churn signals.

### Interim Conclusion
Loyalty indicators should be used as supporting predictors, not as primary drivers, in churn modeling.

## Distribution by Churn Group

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, col in enumerate(Loyalty_features):
    sns.boxplot(
        data=df,
        x="Churned",
        y=col,
        hue="Churned",
        dodge=False,
        legend=False,
        ax=axes[i],
        palette="Set2"
    )
    axes[i].set_title(f"{col} by Churn")

plt.tight_layout()
plt.show()

## Group Summary + Statistical Test

In [ ]:
def rank_biserial_from_u(u_stat, n_churn, n_non_churn):
    auc_probability = u_stat / (n_churn * n_non_churn)
    return 2 * auc_probability - 1


def effect_label(value):
    abs_value = abs(value)
    if abs_value >= 0.5:
        return "Large"
    if abs_value >= 0.3:
        return "Medium"
    if abs_value >= 0.1:
        return "Small"
    return "Negligible"


summary = df.groupby("Churned")[Loyalty_features].mean().T
summary.columns = ["Not Churned (0)", "Churned (1)"]
summary["Diff (0-1)"] = summary["Not Churned (0)"] - summary["Churned (1)"]
summary = summary.sort_values("Diff (0-1)", ascending=False)


test_rows = []
for col in Loyalty_features:
    g0 = df.loc[df["Churned"] == 0, col].dropna()
    g1 = df.loc[df["Churned"] == 1, col].dropna()
    stat, p = stats.mannwhitneyu(g1, g0, alternative="two-sided")
    rb = rank_biserial_from_u(stat, len(g1), len(g0))
    test_rows.append(
        {
            "feature": col,
            "u_stat": stat,
            "p_value": p,
            "rank_biserial": rb,
            "effect_size_label": effect_label(rb),
        }
    )

test_df = pd.DataFrame(test_rows).sort_values("p_value")
loyalty_summary = summary.reset_index().rename(columns={"index": "feature"}).merge(test_df, on="feature", how="left")

print("Executive Insight: Loyalty vs Churn")
print("-----------------------------------")
for _, row in loyalty_summary.iterrows():
    direction = "higher" if row["Diff (0-1)"] < 0 else "lower"
    print(
        f"- {row['feature']}: churn group average is {direction} than non-churn, "
        f"effect={row['effect_size_label']} (rb={row['rank_biserial']:.3f}), p={row['p_value']:.2e}"
    )

loyalty_summary.to_csv("../../../Data/processed/loyalty_professional_summary.csv", index=False)
print("Saved: Data/processed/loyalty_professional_summary.csv")

loyalty_summary